# Notebook 04 — Barren Plateau Proximity Index (BPPI)

**QNLEP metric:** BPPI measures how quickly gradient variance decays with qubit
count. We first fit `log₂(Var[∂L/∂θ])` vs `n`, then map the fitted slope to a
**smooth** plateau-risk score:

```
BPPI = 1 - exp(-2 * max(-α, 0))
```

where `α` is the fitted log-linear slope. This keeps the raw slope interpretable
while avoiding a hard clamp that piles methods up at exact `0` or `1`.

**Reference:** McClean et al. (2018) Nature Comms; Holmes et al. (2022) PRX Quantum.  
**Methods evaluated:** data re-uploading, ancilla tracing, ancilla re-uploading, strongly entangling layers, QCNN, quantum reservoir  
**Outputs:** log(Var) vs n curves, fitted slopes, BPPI bar chart, `bppi_detail.json`

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
import matplotlib.pyplot as plt
from scipy.stats import linregress
import json, pathlib, os

from src.qnlep_registry import QNLEP_BUILDERS, param_shape
from src.metrics.protocol_scores import smooth_bppi_from_slope
from src.qnlep_run_paths import protocol_results_dir, protocol_n_layers

RUN_SEED = int(os.environ.get("QNLEP_SEED", "42"))
np.random.seed(RUN_SEED)
print("QNLEP_SEED =", RUN_SEED)


## 1. Configuration

In [ ]:
N_QUBIT_SWEEP = [2, 4, 6, 8]
N_LAYERS      = protocol_n_layers(2)
N_RANDOM_INIT = 256
x_probe       = 0.5

RESULTS_DIR = protocol_results_dir()
FIGURES_DIR = pathlib.Path('../figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

QNLEP_METHOD_ORDER = ['data_reuploading', 'ancilla_tracing', 'ancilla_reuploading', 'strongly_entangling', 'qcnn', 'quantum_reservoir']

METHOD_BUILDERS = {name: QNLEP_BUILDERS[name] for name in QNLEP_METHOD_ORDER}


## 2. Gradient variance function

In [ ]:
def gradient_variance(builder, method_name, n_qubits, n_layers=N_LAYERS,
                      n_init=N_RANDOM_INIT, x=x_probe):
    """
    Estimate Var_theta[ d/d theta_0 L(theta) ] over random initializations.
    Builds the circuit fresh for each n_qubits via the builder function.
    """
    if method_name == 'quantum_reservoir':
        circuit = builder(n_qubits, n_layers, reservoir_seed=RUN_SEED + n_qubits)
    else:
        circuit = builder(n_qubits, n_layers)
    p_shape = param_shape(method_name, n_qubits, n_layers)

    def loss(w):
        return (circuit(w, x) - 0.0) ** 2

    grad_fn = qml.grad(loss, argnum=0)
    grads = []

    for _ in range(n_init):
        w = pnp.array(np.random.uniform(0, 2 * np.pi, p_shape), requires_grad=True)
        g = grad_fn(w)
        grads.append(float(g.ravel()[0]))

    return float(np.var(grads))


## 3. Sweep: compute Var[grad] for each (method, n_qubits)

In [ ]:
var_data = {}

for method_name, builder in METHOD_BUILDERS.items():
    var_data[method_name] = []
    for n in N_QUBIT_SWEEP:
        v = gradient_variance(builder, method_name, n_qubits=n)
        var_data[method_name].append(v)
        print(f"{method_name:20s}  n={n}  Var[grad] = {v:.2e}")


## 4. Fit log-linear model and compute BPPI

In [ ]:
def compute_bppi(variances, n_values=N_QUBIT_SWEEP):
    """
    Fit log2(Var) = alpha * n + beta via linear regression.
    Export the raw fit, then map the slope to a smooth BPPI score.
    """
    log2_var = np.log2(np.array(variances) + 1e-30)  # guard against log(0)
    slope, intercept, r_value, _, _ = linregress(n_values, log2_var)
    bppi = smooth_bppi_from_slope(slope)
    return bppi, slope, intercept, r_value ** 2

bppi_scores = {}
fit_params  = {}
bppi_detail = {}

print(f"\n{'Method':20s}  {'alpha':>8}  {'R²_fit':>6}  {'BPPI':>6}")
for method_name in METHOD_BUILDERS:
    bppi, slope, intercept, r2_fit = compute_bppi(var_data[method_name])
    bppi_scores[method_name] = bppi
    fit_params[method_name]  = (slope, intercept)
    bppi_detail[method_name] = {
        'slope_alpha': float(slope),
        'intercept_beta': float(intercept),
        'r2_fit': float(r2_fit),
        'gradient_variances': [float(v) for v in var_data[method_name]],
        'score_definition': 'smooth_bppi_from_slope',
    }
    print(f"{method_name:20s}  {slope:8.3f}  {r2_fit:6.3f}  {bppi:6.3f}")


## 5. Visualizations

### 5a. log₂(Var[grad]) vs n_qubits with fitted slopes

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
n_arr = np.array(N_QUBIT_SWEEP)

colors = plt.cm.tab10.colors
for i, (method_name, variances) in enumerate(var_data.items()):
    log2_var = np.log2(np.array(variances) + 1e-30)
    slope, intercept = fit_params[method_name]

    ax.scatter(n_arr, log2_var, color=colors[i], zorder=3, s=60)
    ax.plot(n_arr, slope * n_arr + intercept, color=colors[i], lw=1.5,
            label=f"{method_name}  (α={slope:.2f}, BPPI={bppi_scores[method_name]:.2f})")

# reference line: strong exponential gradient decay slope α = -1
ref_intercept = np.log2(var_data[list(METHOD_BUILDERS.keys())[0]][0])
ax.plot(n_arr, -1.0 * (n_arr - n_arr[0]) + ref_intercept,
        'k--', lw=1, label='strong decay reference (α = -1)')

ax.set_xlabel('Number of qubits')
ax.set_ylabel('log₂ Var[∂L/∂θ]')
ax.set_title('Gradient variance scaling — barren plateau analysis')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/bppi_variance_scaling.pdf', bbox_inches='tight')
plt.show()


### 5b. BPPI bar chart with safe-zone shading

In [ ]:
SAFE_THRESHOLD = 0.3

fig, ax = plt.subplots(figsize=(5, 3))
method_names = list(METHOD_BUILDERS.keys())
bars = ax.bar(method_names, [bppi_scores[m] for m in method_names],
              color=['green' if bppi_scores[m] < SAFE_THRESHOLD else 'tomato'
                     for m in method_names],
              edgecolor='k')

ax.axhline(SAFE_THRESHOLD, color='orange', ls='--', lw=1.5,
           label=f'safe threshold ({SAFE_THRESHOLD})')
ax.axhspan(0, SAFE_THRESHOLD, alpha=0.06, color='green')
ax.set_ylim(0, 1.1)
ax.set_ylabel('BPPI  (smooth plateau-risk score)')
ax.set_title('Barren Plateau Proximity Index')
ax.legend(fontsize=8)
for bar, m in zip(bars, method_names):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{bppi_scores[m]:.2f}", ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../figures/bppi_bar.pdf', bbox_inches='tight')
plt.show()


### 5c. Gradient variance distributions (violin plot at n=4)

Shows the *distribution* of gradients across initializations at a representative
qubit count — variance concentrating near zero is the BP signature.

In [ ]:
N_VIOLIN = 4
grad_distributions = {}

for method_name, builder in METHOD_BUILDERS.items():
    circuit = builder(N_VIOLIN, N_LAYERS)
    grad_fn = qml.grad(lambda w: (circuit(w, x_probe) - 0.0) ** 2, argnum=0)
    samples = []
    for _ in range(N_RANDOM_INIT):
        p_sh = param_shape(method_name, N_VIOLIN, N_LAYERS)
        w = pnp.array(np.random.uniform(0, 2 * np.pi, p_sh), requires_grad=True)
        samples.append(float(grad_fn(w).ravel()[0]))
    grad_distributions[method_name] = samples

fig, ax = plt.subplots(figsize=(5, 4))
ax.violinplot([grad_distributions[m] for m in METHOD_BUILDERS],
              positions=range(len(METHOD_BUILDERS)), showmedians=True)
ax.set_xticks(range(len(METHOD_BUILDERS)))
ax.set_xticklabels(list(METHOD_BUILDERS.keys()), rotation=15, ha='right')
ax.axhline(0, color='k', lw=0.7, ls='--')
ax.set_ylabel('∂L/∂θ₀  (at n=4, random init)')
ax.set_title(f'Gradient distribution at n={N_VIOLIN} qubits')
plt.tight_layout()
plt.savefig('../figures/bppi_violin.pdf', bbox_inches='tight')
plt.show()


## 6. Export BPPI scores

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(RESULTS_DIR / 'bppi_scores.json', 'w', encoding='utf-8') as f:
    json.dump(bppi_scores, f, indent=2)
with open(RESULTS_DIR / 'bppi_detail.json', 'w', encoding='utf-8') as f:
    json.dump({
        'score_definition': 'smooth_bppi_from_slope',
        'n_qubit_sweep': N_QUBIT_SWEEP,
        'n_random_init': N_RANDOM_INIT,
        'x_probe': x_probe,
        'run_seed': RUN_SEED,
        'by_method': bppi_detail,
    }, f, indent=2)
print('Saved to results/bppi_scores.json and results/bppi_detail.json')
